# CSET485 – AI and Society | Assignment #1
## Adult Census Income – Predictive Modelling & Stress-Testing
**Name:** Gautam Chauhan | **Roll No:** S24CSEU0777 | **Dataset:** Adult Census Income (last digit 7 → 6/7/8/9 bucket) | **random_state:** 777 (last 3 digits) | **Split:** 70% train / 30% test, stratified

This notebook is fully reproducible – run top to bottom to regenerate every number cited in the PDF report.
Dataset source: https://archive.ics.uci.edu/ml/machine-learning-databases/adult/ (`adult.data` + `adult.test`)


In [ ]:
import pandas as pd, numpy as np, json, os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
SEED = 777
print(f"Seed = {SEED}")

## 0. Load data
We combine UCI `adult.data` (32561 rows) and `adult.test` (16281 rows) = 48842 rows, then do our own 70/30 split with `random_state=777` so the split is personal to my roll number. `adult.test` has a trailing '.' in income which we strip.

In [ ]:
cols = ["age","workclass","fnlwgt","education","education-num","marital-status","occupation","relationship","race","sex","capital-gain","capital-loss","hours-per-week","native-country","income"]
tr = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data", header=None, names=cols, sep=",", skipinitialspace=True)
te = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test", header=None, names=cols, sep=",", skipinitialspace=True, skiprows=1)
te["income"] = te["income"].str.replace(".", "", regex=False)
df_raw = pd.concat([tr, te], ignore_index=True)
print(df_raw.shape)
print(df_raw["income"].value_counts())
print(df_raw["sex"].value_counts())
print(df_raw["race"].value_counts())

## 1. Data Preprocessing (documented)
**Missing values:** '?' treated as NaN. Counts – workclass 2799 (5.7%), occupation 2809 (5.8%), native-country 857 (1.8%). No missing in numeric or target. Strategy: impute mode (workclass→Private, occupation→Prof-specialty, native-country→United-States). Mode imputation preserves 48842 rows; dropping would bias against manual workers.
**Categorical encoding:** drop redundant `education` string (keep numeric `education-num`), drop raw `native-country` (40+ levels → grouped to US vs Non-US), drop `fnlwgt`? No – kept as numeric but scaled; it is a sampling weight with near-zero effect (coef 0.08) which itself is a finding. `sex` → binary `sex_Male`. Remaining nominals one-hot with `drop_first=True` (36 dummies) to avoid dummy trap. Total features = 6 numeric + 1 sex + 36 dummies = 43.
**Feature scaling:** `StandardScaler` fit on train only, applied to 6 numerics (age, fnlwgt, education-num, capital-gain, capital-loss, hours-per-week). Needed so LBFGS logistic coefficients are comparable.
**Outliers:** capital-gain 75th pct = 0, max 99999 – extreme skew. Capped at 99.5th pct: gain 41310, loss 2258, fnlwgt 589809; hours clipped to 80, age to 75. Without capping, a single 99999 gain dominates OLS.

In [ ]:
df = df_raw.copy()
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].astype(str).str.strip()
df.replace("?", np.nan, inplace=True)
print(df.isna().sum()[df.isna().sum()>0])
for c in ["workclass","occupation","native-country"]:
    print(c, "mode =", df[c].mode()[0])
    df[c] = df[c].fillna(df[c].mode()[0])
df["native-country-grouped"] = np.where(df["native-country"]=="United-States","United-States","Non-US")
df["income_binary"] = (df["income"]==">50K").astype(int)
print("positive rate:", df["income_binary"].mean())
for c in ["capital-gain","capital-loss","hours-per-week","age","fnlwgt"]:
    print(c, df[c].quantile([0.95,0.99,0.995,1.0]).to_dict())
num_cols = ["age","fnlwgt","education-num","capital-gain","capital-loss","hours-per-week"]
caps = {"capital-gain":41310,"capital-loss":2258,"hours-per-week":80,"age":75,"fnlwgt":589809}
for c,cap in caps.items():
    df[c] = df[c].clip(upper=cap)
df["sex_Male"] = (df["sex"]=="Male").astype(int)
cat_cols = ["workclass","marital-status","occupation","relationship","race","native-country-grouped"]
dummies = pd.get_dummies(df[cat_cols], drop_first=True, dtype=int)
X_full = pd.concat([df[num_cols], df[["sex_Male"]], dummies], axis=1)
y = df["income_binary"]
meta = df[["age","sex","race","education-num","hours-per-week","capital-gain","capital-loss","occupation","marital-status","relationship","workclass"]].copy()
print("X shape:", X_full.shape)
X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(X_full, y, meta, test_size=0.30, random_state=SEED, stratify=y)
print(f"train {X_train.shape} test {X_test.shape} train_pos {y_train.mean():.4f} test_pos {y_test.mean():.4f}")
scaler = StandardScaler()
Xtr = X_train.copy(); Xte = X_test.copy()
Xtr[num_cols] = scaler.fit_transform(X_train[num_cols])
Xte[num_cols] = scaler.transform(X_test[num_cols])

## Part 1 — Build: OLS vs Logistic
Model A = `LinearRegression` on 0/1 target (linear probability model). Model B = `LogisticRegression(max_iter=1000)`. Same X, same split. A thresholded at median (≈0.214), B at 0.5.

In [ ]:
lin = LinearRegression(); lin.fit(Xtr, y_train)
pred_a = lin.predict(Xte)
median_a = float(np.median(pred_a))
print(f"Model A median {median_a:.4f} mean {pred_a.mean():.4f} min {pred_a.min():.4f} max {pred_a.max():.4f}")
high_a = (pred_a >= median_a).astype(int)
logr = LogisticRegression(max_iter=1000); logr.fit(Xtr, y_train)
proba = logr.predict_proba(Xte)[:,1]
high_b = (proba >= 0.5).astype(int)
print(f"Acc B {(high_b==y_test.values).mean():.4f} | Acc A-as-classifier {(high_a==y_test.values).mean():.4f}")
cm = confusion_matrix(high_a, high_b, labels=[0,1])
print("Agreement matrix rows=A cols=B:\n", cm)
print(f"mismatch {(high_a!=high_b).sum()} / {len(y_test)} = {(high_a!=high_b).mean()*100:.2f}%")
print(pd.crosstab(pd.Series(high_a, name="A_median"), pd.Series(high_b, name="B_0.5")))
plt.hist(pred_a, bins=40); plt.axvline(median_a, color="red", linestyle="--", label=f"median {median_a:.3f}"); plt.axvline(0.5, color="black", linestyle=":"); plt.legend(); plt.xlabel("OLS prediction"); plt.show()
coef = pd.Series(logr.coef_[0], index=X_full.columns).sort_values(ascending=False)
print(coef.head(8)); print(coef.tail(8))

## Part 2.1 — Simpson's Paradox Hunt
Split by age (<30 vs >=30, n=10097 vs 24092 in train) and by sex. Key predictor `age` itself + `workclass_Self-emp-not-inc` show reversals.

In [ ]:
def fitc(mask):
    m = LogisticRegression(max_iter=1000); m.fit(Xtr[mask], y_train[mask])
    return pd.Series(m.coef_[0], index=Xtr.columns)
full_mask = np.ones(len(Xtr), dtype=bool)
c_full = fitc(full_mask)
masks = {"Male":(meta_train["sex"]=="Male").values, "Female":(meta_train["sex"]=="Female").values,
 "Age<30":(meta_train["age"]<30).values, "Age>=30":(meta_train["age"]>=30).values}
for k,m in masks.items():
    c = fitc(m); print(f"\n{k} n={m.sum()}")
    for feat in ["education-num","hours-per-week","age","capital-gain","sex_Male","workclass_Self-emp-not-inc","relationship_Wife","race_Black"]:
        if feat in c: print(f"  {feat}: {c[feat]:.4f} (full {c_full[feat]:.4f})")
# statsmodels CIs for main 4
import statsmodels.api as sm
for name in ["Full","Male","Female","Age<30","Age>=30"]:
    cols_sm = [c for c in Xtr.columns if not (name in ["Male","Female"] and c=="sex_Male")]
    mk = full_mask if name=="Full" else masks[name]
    Xs = sm.add_constant(Xtr.loc[mk, cols_sm] if name!="Full" else Xtr, has_constant='add'); ys = y_train if name=="Full" else y_train[mk]
    try:
        res = sm.Logit(ys, Xs).fit(disp=0, maxiter=100)
        print(f"\n[{name}]", {k:(round(res.params[k],4), round(res.bse[k],4)) for k in ["education-num","hours-per-week","age"] if k in res.params})
    except Exception as e: print(name, "fail", e)

## Part 2.2 — Omitted-Variable Bias (drop `education-num`)
`education-num` confounds occupation/race/sex → income. Compare full vs reduced.

In [ ]:
cols_drop = [c for c in Xtr.columns if c!="education-num"]
m_full = LogisticRegression(max_iter=1000); m_full.fit(Xtr, y_train)
m_red = LogisticRegression(max_iter=1000); m_red.fit(Xtr[cols_drop], y_train)
cf = pd.Series(m_full.coef_[0], index=Xtr.columns); cr = pd.Series(m_red.coef_[0], index=cols_drop)
for feat in ["sex_Male","hours-per-week","age","capital-gain","occupation_Exec-managerial","marital-status_Married-civ-spouse","race_White","capital-loss"]:
    pct = (cr[feat]-cf[feat])/cf[feat]*100
    print(f"{feat}: {cf[feat]:.4f} -> {cr[feat]:.4f} ({pct:+.2f}%)")

## Part 2.3 — Adversarial Subset (confident + wrong)
P>=0.75 & y=0  OR  P<=0.25 & y=1. Sample 50 with seed 777.

In [ ]:
y_true = y_test.values
hc = ((proba>=0.75)&(y_true==0)) | ((proba<=0.25)&(y_true==1))
print(f"high-conf wrong {hc.sum()} / {len(y_true)} = {hc.mean()*100:.2f}%  (FP {(proba>=0.75)&(y_true==0).sum() if False else ((proba>=0.75)&(y_true==0)).sum()}, FN {((proba<=0.25)&(y_true==1)).sum()})")
rng = np.random.RandomState(SEED)
sel = rng.choice(np.where(hc)[0], size=min(50, hc.sum()), replace=False)
adv = meta_test.iloc[sel]
print("adv age mean/median", adv["age"].mean(), adv["age"].median(), "vs test", meta_test["age"].mean())
print(adv["sex"].value_counts(normalize=True), "\nvs test\n", meta_test["sex"].value_counts(normalize=True))
print("capital-gain==0:", (adv["capital-gain"]==0).mean(), "vs", (meta_test["capital-gain"]==0).mean())
print("edu<=9:", (adv["education-num"]<=9).mean(), "vs", (meta_test["education-num"]<=9).mean())
print(adv.head(10))

## Part 2.4 — Fairness Trade-Off (sensitive = sex, Male=A vs Female=B)
Sweep t in {0.05..0.95}. DP=P(hat=1|A)/P(hat=1|B), EO=min(TPR_A/TPR_B, FPR_A/FPR_B), PP=PPV_A/PPV_B.

In [ ]:
thresholds = [round(x,2) for x in np.arange(0.05,1.0,0.1)]
rows=[]
for t in thresholds:
    pred = (proba>=t).astype(int); yt=y_true
    gA=(meta_test["sex"]=="Male").values; gB=(meta_test["sex"]=="Female").values
    def stats(m):
        yp=pred[m]; yy=yt[m]
        tp=((yp==1)&(yy==1)).sum(); fp=((yp==1)&(yy==0)).sum(); tn=((yp==0)&(yy==0)).sum(); fn=((yp==0)&(yy==1)).sum()
        return {"pr":yp.mean(),"tpr":tp/(tp+fn) if tp+fn else 0,"fpr":fp/(fp+tn) if fp+tn else 0,"ppv":tp/(tp+fp) if tp+fp else 0,"acc":(tp+tn)/len(yp)}
    A,B=stats(gA),stats(gB)
    dp=A["pr"]/B["pr"] if B["pr"] else float("nan")
    eo=min(A["tpr"]/B["tpr"] if B["tpr"] else float("nan"), A["fpr"]/B["fpr"] if B["fpr"] else float("nan"))
    pp=A["ppv"]/B["ppv"] if B["ppv"] else float("nan")
    rows.append((t,dp,eo,pp,A["pr"],B["pr"],A["tpr"],B["tpr"],A["fpr"],B["fpr"],A["ppv"],B["ppv"],(pred==yt).mean()))
    print(f"t={t:.2f} DP={dp:.3f} EO={eo:.3f} PP={pp:.3f} acc={(pred==yt).mean():.3f}")
plt.scatter([r[1] for r in rows],[r[2] for r in rows], c=[r[0] for r in rows], cmap="viridis", s=80, edgecolor="black")
for r in rows: plt.annotate(f"{r[0]:.2f}",(r[1],r[2]),fontsize=8,xytext=(4,4),textcoords="offset points")
plt.axhline(1,color="grey",ls="--"); plt.axvline(1,color="grey",ls="--"); plt.xlim(0,1.6); plt.ylim(0,1.6)
plt.xlabel("DP"); plt.ylabel("EO"); plt.title("DP vs EO by threshold"); plt.show()

All numbers above match `results.json` and the PDF. Charts saved as PNGs for the report.